# Überschrift

## Model

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def draw_network(model):
    G = nx.DiGraph()

    layers = [5, 4, 4, 4, 1]

    positions = {}

    # Create neurons
    for layer_idx, num_neurons in enumerate(layers):
        for neuron_idx in range(num_neurons):
            node = f"{layer_idx}_{neuron_idx}"
            G.add_node(node)

            x = layer_idx
            y = (num_neurons - 1) / 2 - neuron_idx

            positions[node] = (x, y)

    # Create connections
    for layer_idx in range(len(layers) - 1):
        for i in range(layers[layer_idx]):
            for j in range(layers[layer_idx + 1]):
                G.add_edge(
                    f"{layer_idx}_{i}",
                    f"{layer_idx + 1}_{j}"
                )

    plt.figure(figsize=(12, 6))

    nx.draw(
        G,
        positions,
        with_labels=False,
        node_size=1000,
        arrows=True,
        arrowsize=15,
        width=1
    )

    plt.axis("off")
    plt.show()


draw_network(model)

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(5, 4),
    nn.ReLU(), # max(0,x)

    nn.Linear(4, 4),
    nn.ReLU(),

    nn.Linear(4, 4),
    nn.ReLU(),

    nn.Linear(4, 1),
)

## Traningsset

In [ ]:
import pandas as pd

data = pd.read_csv("trainingsset.csv")
data.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns="y").values
y = data["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

## Train

In [ ]:
loss_fn = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
for epoch in range(50_000):

    # ---- TRAIN ----

    model.train()

    prediction = model(X_train) 

    train_loss = loss_fn(prediction, y_train)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()


    # ---- VALIDATE ----

    model.eval()

    with torch.no_grad():
        test_prediction = model(X_test)
        test_loss = loss_fn(
            test_prediction,
            y_test
        )


    if epoch % 1_000 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"train loss: {train_loss.item():.1f} | "
            f"test loss: {test_loss.item():.1f}"
        )

In [ ]:
with torch.no_grad():
    prediction = model(X_test)

    percentage_error = (
        torch.abs((y_test - prediction) / y_test) * 100
    )

    mape = percentage_error.mean()

print(f"MAPE: {mape.item():.2f}%")

## Export Weights

In [ ]:
torch.save(model.state_dict(), "bevoelkerungsstand_model.pth")